# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense




Follow these steps to prepare the Tic-Tac-Toe data for a neural network that predicts whether Player X wins.

## Step 1: Data Engineering

1. Load `tic-tac-toe.csv` into a pandas DataFrame.
2. Inspect the rows, columns, data types, and missing values.
3. Encode each board position as a number: `x` = 1, `o` = -1, and `b` (blank) = 0. Convert the boolean target `class` to 1 for `True` (Player X wins) and 0 for `False` (Player X does not win).
4. Separate the nine input features (`X`) from the target (`y`).
5. Scale the input features to the range 0 to 1. Keep the target as 0 or 1; it is already numeric after conversion.

In [2]:
from pathlib import Path
import pandas as pd

csv_candidates = [
    Path("tic-tac-toe.csv"),
    Path("your-code/tic-tac-toe.csv"),
    Path("/content/tic-tac-toe.csv"),
    Path("/content/lab-neural-networks/your-code/tic-tac-toe.csv"),
]
csv_path = next((path for path in csv_candidates if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "Could not find tic-tac-toe.csv. Place it beside the notebook or in the your-code folder."
    )

df = pd.read_csv(csv_path)
df.head()

,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,x,x,x,x,o,o,x,o,o,True
1,x,x,x,x,o,o,o,x,o,True
2,x,x,x,x,o,o,o,o,x,True
3,x,x,x,x,o,o,o,b,b,True
4,x,x,x,x,o,o,b,o,b,True


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   TL      958 non-null    object
 1   TM      958 non-null    object
 2   TR      958 non-null    object
 3   ML      958 non-null    object
 4   MM      958 non-null    object
 5   MR      958 non-null    object
 6   BL      958 non-null    object
 7   BM      958 non-null    object
 8   BR      958 non-null    object
 9   class   958 non-null    bool  
dtypes: bool(1), object(9)
memory usage: 68.4+ KB


The dataset contains 958 game positions, nine board-position features, and one target column (`class`). All columns have 958 non-null values. The board positions are categorical text (`x`, `o`, or `b`), while `class` is boolean: `True` means Player X wins; `False` means Player X does not win (Player O wins or the game is a draw). Neural-network training code expects numeric inputs and numeric class labels, so I encode both before modeling.

In [4]:
df.head()

,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,x,x,x,x,o,o,x,o,o,True
1,x,x,x,x,o,o,o,x,o,True
2,x,x,x,x,o,o,o,o,x,True
3,x,x,x,x,o,o,o,b,b,True
4,x,x,x,x,o,o,b,o,b,True


In [5]:
df.sample(5)

,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
534,b,o,x,o,x,x,x,o,b,True
532,b,o,x,x,o,x,o,b,x,True
914,b,o,x,x,o,x,x,o,o,False
5,x,x,x,x,o,o,b,b,o,True
432,o,b,x,o,x,x,x,b,o,True


In [6]:
df.shape

(958, 10)

First I inspect the dataset's size and contents. I will separate `X` (the nine board positions) and `y` (the win label) after encoding, so both are ready for the model.

In [7]:
df.isna().sum()

,0
TL,0
TM,0
TR,0
ML,0
MM,0
MR,0
BL,0
BM,0
BR,0
class,0


The board columns contain categorical text (`x`, `o`, and `b`), so they must be encoded as numbers before training as mentoned before. The target column is boolean, so it also needs conversion to numeric labels: `True` becomes 1 and `False` becomes 0.

In [8]:
df.head()


,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,x,x,x,x,o,o,x,o,o,True
1,x,x,x,x,o,o,o,x,o,True
2,x,x,x,x,o,o,o,o,x,True
3,x,x,x,x,o,o,o,b,b,True
4,x,x,x,x,o,o,b,o,b,True


In [9]:

df.head()
df.info()
df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   TL      958 non-null    object
 1   TM      958 non-null    object
 2   TR      958 non-null    object
 3   ML      958 non-null    object
 4   MM      958 non-null    object
 5   MR      958 non-null    object
 6   BL      958 non-null    object
 7   BM      958 non-null    object
 8   BR      958 non-null    object
 9   class   958 non-null    bool  
dtypes: bool(1), object(9)
memory usage: 68.4+ KB


,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
count,958,958,958,958,958,958,958,958,958,958
unique,3,3,3,3,3,3,3,3,3,2
top,x,x,x,x,x,x,x,x,x,True
freq,418,378,418,378,458,378,418,378,418,626


In [10]:
for col in df.columns[:-1]:
    print(df[col].unique())


['x' 'o' 'b']
['x' 'o' 'b']
['x' 'o' 'b']
['x' 'o' 'b']
['o' 'b' 'x']
['o' 'b' 'x']
['x' 'o' 'b']
['o' 'x' 'b']
['o' 'x' 'b']


The initial inspection looks reasonable: there are 958 rows, no missing values, nine board-position columns, and one target column. The board columns contain only the expected values (`x`, `o`, and `b`), and the target is boolean. This is a useful basic check that the file loaded as expected; it does not by itself prove every game position is valid.

In [11]:
board_columns = df.columns.drop("class")

mapping = {
    "x": 1,
    "o": -1,
    "b": 0,
}
df[board_columns] = df[board_columns].replace(mapping).astype("int8")
df["class"] = df["class"].astype("int8")

assert df[board_columns].notna().all().all()
assert df["class"].isin([0, 1]).all()
df.head()

/tmp/ipykernel_78850/2975798157.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[board_columns] = df[board_columns].replace(mapping).astype("int8")


,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,1,1,1,1,-1,-1,1,-1,-1,1
1,1,1,1,1,-1,-1,-1,1,-1,1
2,1,1,1,1,-1,-1,-1,-1,1,1
3,1,1,1,1,-1,-1,-1,0,0,1
4,1,1,1,1,-1,-1,0,-1,0,1


In [12]:
X = df.drop(columns="class")
y = df["class"]

# Scale board values from -1, 0, 1 to 0, 0.5, 1.
X = (X + 1) / 2

print("Input shape:", X.shape)
print("Target shape:", y.shape)
print("Input value range:", X.min().min(), "to", X.max().max())
print("Target values:", sorted(y.unique()))
X.head()

Input shape: (958, 9)
Target shape: (958,)
Input value range: 0.0 to 1.0
Target values: [np.int8(0), np.int8(1)]


,TL,TM,TR,ML,MM,MR,BL,BM,BR
0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
1,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
2,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
3,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.5,0.5
4,1.0,1.0,1.0,1.0,0.0,0.0,0.5,0.0,0.5


## Step 2: Build the Neural Network

1. Split `X` and `y` into training and test sets, keeping the class proportions similar in both sets.
2. Create a `Sequential` model with an input layer for the nine board positions.
3. Add hidden layers with ReLU activation and a two-unit Softmax output layer. The two output values represent the probabilities that Player X does not win or wins.
4. Compile the model with the Adam optimizer, `sparse_categorical_crossentropy` loss, and accuracy as a metric. This loss works with the integer class labels 0 and 1.
5. Fit the model using the training data.
6. Evaluate its loss and accuracy on the held-out test data.
7. Save the trained model as `tic-tac-toe.keras`, Keras's supported model-file format.

In [13]:
from sklearn.model_selection import train_test_split

# Keep the class proportions similar in the training and test sets.
tf.keras.utils.set_random_seed(42)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

model = Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(2, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
)

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

model_path = "tic-tac-toe.keras"
model.save(model_path)
print(f"Model saved to {model_path}")

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6585 - loss: 0.6314 - val_accuracy: 0.6364 - val_loss: 0.6383
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6585 - loss: 0.6122 - val_accuracy: 0.6364 - val_loss: 0.6285
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6585 - loss: 0.5983 - val_accuracy: 0.6364 - val_loss: 0.6165
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6585 - loss: 0.5835 - val_accuracy: 0.6364 - val_loss: 0.6010
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6683 - loss: 0.5668 - val_accuracy: 0.6494 - val_loss: 0.5848
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6993 - loss: 0.5507 - val_accuracy: 0.6753 - val_loss: 0.5693
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7337 - loss: 0.5358 - val_accuracy: 0.6818 - val_loss: 0.5562
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7451 - loss: 0.5226 - val_accuracy: 0.7208 - 

## Step 3: Make Predictions

Load the saved model and predict the labels for a few test examples. Here, `0` means Player X does not win and `1` means Player X wins. Compare each prediction with the actual label to see whether it is correct.

In [17]:
loaded_model = tf.keras.models.load_model(model_path)

sample_inputs = X_test.sample(n=5, random_state=42)
sample_probabilities = loaded_model.predict(sample_inputs, verbose=0)
predicted_classes = sample_probabilities.argmax(axis=1)

prediction_results = sample_inputs.copy()
prediction_results["actual"] = y_test.loc[sample_inputs.index]
prediction_results["predicted"] = predicted_classes
prediction_results["correct"] = prediction_results["actual"] == prediction_results["predicted"]
prediction_results

,TL,TM,TR,ML,MM,MR,BL,BM,BR,actual,predicted,correct
301,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.5,0.5,1,1,True
416,0.0,0.0,0.5,1.0,0.5,0.0,1.0,1.0,1.0,1,1,True
89,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1,1,True
392,0.0,0.0,1.0,0.5,1.0,1.0,1.0,0.0,0.5,1,1,True
57,1.0,1.0,1.0,0.5,0.0,0.0,1.0,0.5,0.0,1,1,True


## Step 4: Improve Your Model

The baseline model is evaluated against the goals of test loss below 0.1 and test accuracy above 0.95. Experiment with the number of layers, learning rate, and training epochs, then compare the new model's test results with the baseline. Early stopping can restore the weights from the epoch with the best validation loss.

In [18]:
from tensorflow.keras.callbacks import EarlyStopping

# Try a wider network and a smaller learning rate than the baseline model.
tf.keras.utils.set_random_seed(42)
improved_model = Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(2, activation="softmax"),
])
improved_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

stop_early = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
)
improved_history = improved_model.fit(
    X_train,
    y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=[stop_early],
    verbose=0,
)

improved_loss, improved_accuracy = improved_model.evaluate(X_test, y_test, verbose=0)
print(f"Training stopped after {len(improved_history.history['loss'])} epochs")
print(f"Baseline test loss: {test_loss:.4f}; accuracy: {test_accuracy:.4f}")
print(f"Improved test loss: {improved_loss:.4f}; accuracy: {improved_accuracy:.4f}")

improved_model_path = "tic-tac-toe-improved.keras"
improved_model.save(improved_model_path)
print(f"Improved model saved to {improved_model_path}")

Training stopped after 200 epochs
Baseline test loss: 0.0986; accuracy: 0.9688
Improved test loss: 0.0993; accuracy: 0.9844
Improved model saved to tic-tac-toe-improved.keras


**Which approach(es) helped improve model performance?**

The baseline model already met both targets: test loss was 0.0986 and test accuracy was 0.9688. I tried a wider network with an additional hidden layer, a lower learning rate of 0.0005, and up to 200 epochs. The improved model reached 0.9844 test accuracy, while test loss was 0.0993. Accuracy improved slightly, but loss was nearly unchanged and increased by 0.0007, so the new model is not better on every metric. Both models meet the stated targets.

In [19]:
print(f"Baseline test loss: {test_loss:.4f}; accuracy: {test_accuracy:.4f}")
print(f"Improved test loss: {improved_loss:.4f}; accuracy: {improved_accuracy:.4f}")

Baseline test loss: 0.0986; accuracy: 0.9688
Improved test loss: 0.0993; accuracy: 0.9844


Overall!!!!A wider network, a lower learning rate, and longer training improved test accuracy from 0.9688 to 0.9844. Test loss changed slightly from 0.0986 to 0.0993, so accuracy improved while loss remained below 0.1. Since I changed several settings together, I cannot tell which change helped the most by itself.